In [1]:
import os, sys, copy; from pathlib import Path; os.chdir(Path.home()/"IAA"); sys.path.insert(0, str(Path.home()/"IAA"))
import numpy as np, pandas as pd
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import mean_tweedie_deviance
import src.pipeline as P, src.target as T
from src.config import PART2
from src.evaluate import evaluate

out = Path("results/bootstrap_murphy"); out.mkdir(parents=True, exist_ok=True)

# ---------- 1. 2024 validation rows with the v7 structure
cfg = copy.deepcopy(PART2)
T.AMOUNTS_NET, T.ROW_COMPLETENESS = bool(cfg["amounts_net"]), bool(cfg["row_completeness"])
claims = T.load_claims(cfg["data_dir"])
parts = []
for line in cfg["lines"]:
    d = P.build_line(line, cfg, claims)
    tr0 = d["tr"].merge(d["p"][["row_id", "policy_id"]], on="row_id")
    vy, tys = cfg["folds"][0]
    tr, va = tr0[tr0.year.isin(tys)], tr0[tr0.year == vy]
    rate, _ = P.predict_rate(line, cfg, tr, va, d["feats"])
    rate = np.asarray(rate, float) + d.get("sep_rate", 0.0)
    parts.append(pd.DataFrame({"line": line, "row_id": va.row_id.to_numpy(), "policy_id": va.policy_id.astype(str).to_numpy(),
                               "nb": va.new_business.astype(bool).to_numpy(), "w": va.weight.to_numpy(),
                               "pv": rate * va.weight.to_numpy(),
                               "act": (va.target_all if "target_all" in va else va.target).to_numpy()}))
V = pd.concat(parts, ignore_index=True)
V.to_csv(out / "valid2024_rows_v7.csv", index=False)
print(f"rows {len(V):,} | pooled D2 {evaluate(V.act, V.pv, V.w, show=False)['D2']:.4f}\n")

# ---------- 2. Murphy decomposition (Tweedie p=1.5, same baseline as evaluate: one rate x weight)
def S(a, p): return mean_tweedie_deviance(a, np.maximum(p, 1e-9), power=1.5)
def murphy(df):
    a, p, w = df.act.to_numpy(), df.pv.to_numpy(), df.w.to_numpy()
    base = a.sum() / w.sum() * w
    r = p / w
    iso = IsotonicRegression(increasing=True, out_of_bounds="clip").fit(r, a / w, sample_weight=w)
    rec = np.maximum(iso.predict(r), 0) * w                                   # best monotone recalibration
    kstar = (a / np.sqrt(np.maximum(p, 1e-9))).sum() / np.sqrt(np.maximum(p, 1e-9)).sum()
    s, su, sr, sl = S(a, p), S(a, base), S(a, rec), S(a, p * kstar)
    return {"rows": len(a), "D2": 1 - s / su, "DSC (ranking)": (su - sr) / su, "MCB (miscalibration)": (s - sr) / su,
            "MCB removed by one level factor": (s - sl) / su, "k* (Tweedie-optimal level)": kstar, "A/E": a.sum() / p.sum()}
mrows = []
for line in ["ALL", "motor", "buildings", "contents"]:
    dl = V if line == "ALL" else V[V.line == line]
    for seg, ds in [("all rows", dl), ("new business", dl[dl.nb]), ("renewal", dl[~dl.nb])]:
        mrows.append({"line": line, "segment": seg, **murphy(ds)})
M = pd.DataFrame(mrows); M.to_csv(out / "murphy.csv", index=False)
pd.set_option("display.width", 220)
print("=== Murphy decomposition (shares of the constant-model deviance; D2 = DSC - MCB) ===")
print(M.round(4).to_string(index=False), "\n")

# ---------- 3. bootstrap of k* over random policy subsets, scaled to the test public size
TEST_ROWS = {("motor", True): 68823, ("motor", False): 21698, ("buildings", True): 39716,
             ("buildings", False): 12579, ("contents", True): 50441, ("contents", False): 15806}
rng, B, Q = np.random.default_rng(2026), 2000, 0.5
V["u"] = V.act / np.sqrt(V.pv.clip(lower=1e-9)); V["c"] = np.sqrt(V.pv.clip(lower=1e-9))
brows = []
for line in ["ALL", "motor", "buildings", "contents"]:
    dl = V if line == "ALL" else V[V.line == line]
    for seg, nbflag in [("all rows", None), ("new business", True), ("renewal", False)]:
        ds = dl if nbflag is None else dl[dl.nb == nbflag]
        g = ds.groupby("policy_id")[["u", "c"]].sum(); U, C = g.u.to_numpy(), g.c.to_numpy()
        ks = np.array([(lambda m: U[m].sum() / C[m].sum())(rng.random(len(U)) < Q) for _ in range(B)])
        lines_ = ["motor", "buildings", "contents"] if line == "ALL" else [line]
        flags = [True, False] if nbflag is None else [nbflag]
        n_test = sum(TEST_ROWS[(l, f)] for l in lines_ for f in flags)
        row = {"line": line, "segment": seg, "validation rows": len(ds), "k* (full fold)": U.sum() / C.sum(),
               "sd at half the fold": ks.std(), "95% at half the fold": f"{np.percentile(ks, 2.5):.3f}-{np.percentile(ks, 97.5):.3f}"}
        for f in (0.2, 0.3, 0.5):                                             # public share unknown
            sd = ks.std() * np.sqrt(Q * len(ds) / (f * n_test))
            row[f"+-95% at public {int(f * 100)}%"] = 1.96 * sd
        brows.append(row)
Bt = pd.DataFrame(brows); Bt.to_csv(out / "bootstrap_kstar.csv", index=False)
print("=== bootstrap of the Tweedie-optimal scale k* (relative spread; scaled to the test public size) ===")
print(Bt.round(4).to_string(index=False))
nb = Bt[(Bt.line == "ALL") & (Bt.segment == "new business")].iloc[0]
print(f"\nnew-business load 1.29 (leaderboard): public-sampling 95% band about "
      + ", ".join(f"{1.29 * (1 - nb[c]):.2f}-{1.29 * (1 + nb[c]):.2f} ({c.split('at ')[1]})" for c in Bt.columns if c.startswith("+-95%")))
print(f"\nsaved to {out}/")

23:01:55 motor: developed training table 979,968 rows
23:02:20 buildings: sum_insured x1000 entries divided by 1000: train 53, test 104
23:02:20 buildings: fire priced separately at 97.05 per exposure (developed)
23:02:20 buildings: own GLM for ['flood']; main GLM on the other perils
23:02:25 buildings: developed training table 707,726 rows
23:02:32 contents: sum_insured x1000 entries divided by 1000: train 57, test 135
23:02:36 contents: developed training table 721,617 rows
rows 59,202 | pooled D2 0.1081

=== Murphy decomposition (shares of the constant-model deviance; D2 = DSC - MCB) ===
     line      segment  rows      D2  DSC (ranking)  MCB (miscalibration)  MCB removed by one level factor  k* (Tweedie-optimal level)    A/E
      ALL     all rows 59202  0.1081         0.1166                0.0085                           0.0000                      1.0067 1.0050
      ALL new business  9063  0.0652         0.0901                0.0250                           0.0001            

In [2]:
import os; from pathlib import Path; os.chdir(Path.home()/"IAA")
txt = """### Murphy decomposition and bootstrap of the new-business load (Oct 24, 23:03, notebook 29) — results/bootstrap_murphy/
- 2024 validation rows rebuilt with the v7 structure (pooled D2 0.1081). Murphy decomposition of the Tweedie p=1.5 score (shares of the
  constant-model deviance; recalibration by weighted isotonic regression in-sample, so MCB is an upper bound):
  all rows D2 0.108 = DSC 0.117 - MCB 0.0085, of which one level factor removes 0.0000; new business D2 0.065 = DSC 0.090 - MCB 0.025;
  renewals D2 0.114 = DSC 0.124 - MCB 0.010. The level is right in validation; new business ranks less well (no claim history), which is
  76% of the test. Buildings / contents new-business MCB 0.079 / 0.072 on 35 / 64 losses: small-sample noise.
- Bootstrap of the Tweedie-optimal scale k* (2,000 random halves of the fold by policy, spread scaled to the test public size; public share
  unknown): new-business load 1.29 has a public-sampling 95% band of 1.19-1.39 (public 20%), 1.21-1.37 (30%), 1.23-1.35 (50%).
  x1.3 sits in the middle; tuning within 1.25-1.35 is below sampling noise. The band covers public sampling only, not whether the broker
  book is really 1.3x (that uncertainty is separate and larger). Approximation: validation rows scaled to test size.
"""
p = Path("DECISIONS.md"); d = p.read_text(encoding="utf-8")
a = "## Assumptions register"; assert d.count(a) == 1 and "Murphy decomposition and bootstrap" not in d
p.write_text(d.replace(a, txt + "\n" + a), encoding="utf-8"); print("recorded")

recorded
